# 01 - Bronze Ingest (CSV → Delta)

In [ ]:
dbutils.widgets.text("raw_base", "abfss://olistdata@olistecommdatastorage.dfs.core.windows.net/bronze")
dbutils.widgets.text("curated_base", "abfss://olistdata@olistecommdatastorage.dfs.core.windows.net/curated")
RAW = dbutils.widgets.get("raw_base").rstrip("/")
CUR = dbutils.widgets.get("curated_base").rstrip("/")

In [ ]:
from pyspark.sql.types import *; from pyspark.sql.functions import to_timestamp,to_date,year,month
orders_schema=StructType([StructField("order_id",StringType()),StructField("customer_id",StringType()),StructField("order_status",StringType()),StructField("order_purchase_timestamp",StringType()),StructField("order_approved_at",StringType()),StructField("order_delivered_carrier_date",StringType()),StructField("order_delivered_customer_date",StringType()),StructField("order_estimated_delivery_date",StringType())])
sources={"customers":("olist_customers_dataset.csv",None),"geolocation":("olist_geolocation_dataset.csv",None),"order_items":("olist_order_items_dataset.csv",None),"order_payments":("olist_order_payments_dataset.csv",None),"order_reviews":("olist_order_reviews_dataset.csv",None),"orders":("olist_orders_dataset.csv",orders_schema),"products":("olist_products_dataset.csv",None),"sellers":("olist_sellers_dataset.csv",None),"product_category_name_translation":("product_category_name_translation.csv",None)}
for table,(file,schema) in sources.items():
  reader=spark.read.option("header",True).option("multiLine",True)
  if schema: reader=reader.schema(schema)
  df=reader.csv(f"{RAW}/{file}")
  if table=="orders":
    df=(df.withColumn("order_ts",to_timestamp("order_purchase_timestamp")).withColumn("order_date",to_date("order_ts")).withColumn("order_year",year("order_date")).withColumn("order_month",month("order_date")))
    (df.write.format("delta").mode("overwrite").partitionBy("order_year","order_month").save(f"{CUR}/bronze/{table}"))
  else:
    (df.write.format("delta").mode("overwrite").save(f"{CUR}/bronze/{table}"))
print("Batch bronze ingestion complete.")

In [ ]:
# %run ./utils  # uncomment in Databricks

In [ ]:
# from pyspark.sql.functions import to_timestamp, to_date, year, month

# sources = {
#     "customers": "olist_customers_dataset.csv",
#     "geolocation": "olist_geolocation_dataset.csv",
#     "order_items": "olist_order_items_dataset.csv",
#     "order_payments": "olist_order_payments_dataset.csv",
#     "order_reviews": "olist_order_reviews_dataset.csv",
#     "orders": "olist_orders_dataset.csv",
#     "products": "olist_products_dataset.csv",
#     "sellers": "olist_sellers_dataset.csv",
#     "product_category_name_translation": "product_category_name_translation.csv"
# }

# for table, file in sources.items():
#     df = (spark.read
#             .option("header", True)
#             .option("multiLine", True)
#             .csv(f"{RAW}/{file}"))

#     if table == "orders":
#         df = (df
#             .withColumn("order_ts", to_timestamp("order_purchase_timestamp"))
#             .withColumn("order_date", to_date("order_ts"))
#             .withColumn("order_year", year("order_date"))
#             .withColumn("order_month", month("order_date")))
#         (df.write.format("delta").mode("overwrite")
#             .partitionBy("order_year","order_month")
#             .save(f"{CUR}/bronze/{table}"))
#     else:
#         (df.write.format("delta").mode("overwrite")
#             .save(f"{CUR}/bronze/{table}"))

# print("Bronze ingestion complete.")

In [ ]:
# orders_bz = spark.read.format("delta").load(f"{CUR}/bronze/orders")
# print("Orders bronze sample:")
# orders_bz.show(10, truncate=False)